# 🎙️ Discussion Stream Analyzer - Colab launcher
1. **Runtime > Change runtime type > T4 GPU**
2. Set `REPO_URL` below to your GitHub repository
3. *Run all*, then open the printed `trycloudflare.com` link and keep the last cell running.

The link is public: don't share it.

In [ ]:
REPO_URL = "https://github.com/<your-username>/discussion-stream-analyzer.git"   # <- change this
import os
os.chdir("/content")
if not os.path.exists("discussion-stream-analyzer"):
    !git clone -q {REPO_URL} discussion-stream-analyzer
os.chdir("/content/discussion-stream-analyzer")
!pip install -q -r requirements.txt

In [ ]:
import socket, subprocess, time
subprocess.run("pkill -f streamlit; pkill -f cloudflared", shell=True)
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared", shell=True, check=True)
subprocess.Popen("nohup streamlit run app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit_logs.txt 2>&1 &", shell=True)
for _ in range(60):                       # wait until the app answers instead of a fixed sleep
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", 8501)) == 0:
            break
    time.sleep(1)
!/content/cloudflared tunnel --url http://localhost:8501

If the page doesn't load, check the app log:

In [ ]:
!tail -n 40 /content/streamlit_logs.txt